# German Insolvency Analysis (2020–2025)
**Author:** Maria Schulmann  
**Purpose:** Data pipeline for cleaning and preparing German insolvency statistics for Tableau visualization

This notebook processes raw data from [Destatis](https://www.destatis.de) (Federal Statistical Office of Germany) and produces clean, analysis-ready CSV files covering:

- **National trends** — monthly insolvency filings and outstanding claims across Germany
- **Regional breakdown** — insolvencies by Bundesland (federal state)
- **Industry sectors** — filings by economic activity (WZ08 classification), enriched with business register data
- **Company size** — insolvencies segmented by employee count

**Pipeline:** Raw Destatis CSVs → Python/Pandas cleaning → Clean CSVs → Tableau Public dashboard

## 1. Setup & Imports

In [ ]:
import pandas as pd
import numpy as np
import os
import sys

print(f"Python {sys.version.split()[0]} | Pandas {pd.__version__} | NumPy {np.__version__}")

## 2. Configuration

German month names mapped to numeric values, used throughout the cleaning pipeline.

In [ ]:
DATA_FOLDER = "data"
CLEAN_FOLDER = "data/clean"
os.makedirs(CLEAN_FOLDER, exist_ok=True)

MONTH_MAP = {
    "Januar": "01", "Februar": "02", "März": "03",
    "April": "04", "Mai": "05", "Juni": "06",
    "Juli": "07", "August": "08", "September": "09",
    "Oktober": "10", "November": "11", "Dezember": "12"
}

## 3. Load Raw Data

The Destatis CSV files have German headers and several metadata rows at the top. We skip the first 6 rows to reach the actual data.

| File | Content |
|------|---------|
| `52411-0001` | Monthly insolvency totals for all of Germany |
| `52411-0014` | Corporate insolvencies with affected employee counts |
| `52411-0018` | Corporate insolvencies by industry sector (WZ08) |
| `52411-0013` | Corporate insolvencies by company size class |
| `52411-0110` | Corporate insolvencies by Bundesland (federal state) |
| `52111-0003` | Business register — active companies by industry |

In [ ]:
df_germany    = pd.read_csv(f"{DATA_FOLDER}/52411-0001_de.csv", sep=";", skiprows=6, encoding="utf-8-sig")
df_industry   = pd.read_csv(f"{DATA_FOLDER}/52411-0018_de.csv", sep=";", skiprows=6, encoding="utf-8-sig")
df_size       = pd.read_csv(f"{DATA_FOLDER}/52411-0013_de.csv", sep=";", skiprows=6, encoding="utf-8-sig")
df_corporate  = pd.read_csv(f"{DATA_FOLDER}/52411-0014_de.csv", sep=";", skiprows=6, encoding="utf-8-sig", header=None)

print("Raw files loaded:")
for name, df in [("df_germany", df_germany), ("df_industry", df_industry),
                 ("df_size", df_size), ("df_corporate", df_corporate)]:
    print(f"  {name:20s} {df.shape[0]:>5} rows x {df.shape[1]:>3} cols")

## 4. Clean & Prepare Data

### 4a. National Totals (`germany_clean`)

Monthly insolvency count, year-over-year change, and outstanding claims for all of Germany.

In [ ]:
def clean_germany(df):
    df = df.copy()
    df.columns = ["year", "month", "insolvencies", "ins_flag",
                  "yoy_change_pct", "yoy_flag", "claims_tsd_eur", "claims_flag"]
    
    df["year"] = df["year"].ffill()
    df = df[df["month"].isin(MONTH_MAP.keys())].copy()
    
    df["month_num"] = df["month"].map(MONTH_MAP)
    df["date"] = pd.to_datetime(df["year"].astype(str) + "-" + df["month_num"])
    
    for col in ["insolvencies", "claims_tsd_eur"]:
        df[col] = pd.to_numeric(df[col], errors="coerce")
    df["yoy_change_pct"] = pd.to_numeric(
        df["yoy_change_pct"].astype(str).str.replace(",", ".").str.strip(), errors="coerce"
    )
    
    df = df[["date", "year", "month", "month_num",
             "insolvencies", "yoy_change_pct", "claims_tsd_eur"]].copy()
    df["year"] = df["year"].astype(int)
    return df.reset_index(drop=True)

germany_clean = clean_germany(df_germany)
print(f"germany_clean: {germany_clean.shape[0]} rows, {germany_clean['date'].min():%Y-%m} to {germany_clean['date'].max():%Y-%m}")
germany_clean.head()

### 4b. Industry Breakdown (`industry_clean`)

Monthly insolvency counts by WZ08 industry classification. The raw file uses a hierarchical row structure with year → month → industry code rows, which we parse iteratively.

In [ ]:
def clean_industry(df):
    df = df.copy()
    df.columns = ["code_or_date", "industry", "opened", "opened_flag",
                  "dismissed", "dismissed_flag", "debt_plan", "debt_plan_flag"]
    
    current_year, current_month = None, None
    records = []

    for _, row in df.iterrows():
        val = str(row["code_or_date"]).strip()
        
        if val.isdigit() and len(val) == 4:
            current_year = int(val)
            continue
        if val in MONTH_MAP:
            current_month = val
            continue
        if val.startswith("WZ08-") and current_year and current_month:
            opened = pd.to_numeric(row["opened"], errors="coerce")
            dismissed = pd.to_numeric(row["dismissed"], errors="coerce")
            records.append({
                "date": pd.to_datetime(f"{current_year}-{MONTH_MAP[current_month]}"),
                "year": current_year,
                "month": current_month,
                "month_num": MONTH_MAP[current_month],
                "industry_code": val,
                "industry": str(row["industry"]).strip(),
                "opened": opened,
                "dismissed": dismissed,
                "total": (opened or 0) + (dismissed or 0),
            })

    return pd.DataFrame(records).reset_index(drop=True)

industry_clean = clean_industry(df_industry)
print(f"industry_clean: {industry_clean.shape[0]} rows, {industry_clean['industry_code'].nunique()} industries")
industry_clean.head()

### 4c. Company Size Breakdown (`size_clean`)

Insolvencies segmented by number of employees. Follows the same hierarchical row parsing approach.

In [ ]:
def clean_size(df):
    df = df.copy()
    df.columns = ["size_class", "insolvencies", "ins_flag",
                  "claims_tsd_eur", "claims_flag",
                  "employees_affected", "emp_flag"]
    
    SIZE_CLASSES = [
        "1 Arbeitnehmer", "2 bis 5 Arbeitnehmer", "6 bis 10 Arbeitnehmer",
        "11 bis 100 Arbeitnehmer", "Über 100 Arbeitnehmer",
        "Unbekannt oder kein Arbeitnehmer",
    ]
    
    current_year, current_month = None, None
    records = []

    for _, row in df.iterrows():
        val = str(row["size_class"]).strip()
        if val.isdigit() and len(val) == 4:
            current_year = int(val)
            continue
        if val in MONTH_MAP:
            current_month = val
            continue
        if val in SIZE_CLASSES and current_year and current_month:
            records.append({
                "date": pd.to_datetime(f"{current_year}-{MONTH_MAP[current_month]}"),
                "year": current_year,
                "month": current_month,
                "month_num": MONTH_MAP[current_month],
                "size_class": val,
                "insolvencies": pd.to_numeric(row["insolvencies"], errors="coerce"),
                "claims_tsd_eur": pd.to_numeric(row["claims_tsd_eur"], errors="coerce"),
                "employees_affected": pd.to_numeric(row["employees_affected"], errors="coerce"),
            })

    return pd.DataFrame(records).reset_index(drop=True)

size_clean = clean_size(df_size)
print(f"size_clean: {size_clean.shape[0]} rows, {size_clean['size_class'].nunique()} size classes")
size_clean.head()

### 4d. Regional Breakdown (`bundesland_clean`)

The Bundesland file is the trickiest to parse — it's a wide-format CSV where each of the 16 states occupies 8 columns (value + flag pairs for insolvencies, YoY change, claims, and employees). We extract state names from the header row and pivot into long format.

In [ ]:
def clean_bundesland(filepath):
    header_raw = pd.read_csv(filepath, sep=";", encoding="utf-8-sig",
                              header=None, skiprows=4, nrows=1,
                              on_bad_lines="skip")
    
    state_names = []
    for i in range(2, header_raw.shape[1], 8):
        val = str(header_raw.iloc[0, i]).strip()
        if val and val not in ("nan", ""):
            state_names.append((i, val))
    
    data_raw = pd.read_csv(filepath, sep=";", encoding="utf-8-sig",
                            header=None, skiprows=6, on_bad_lines="skip")
    data_raw[0] = data_raw[0].ffill()
    data_raw = data_raw[pd.to_numeric(data_raw[0], errors="coerce").notna()]
    data_raw = data_raw[data_raw[1].isin(MONTH_MAP.keys())]

    records = []
    for _, row in data_raw.iterrows():
        year = str(row[0]).strip()
        month = str(row[1]).strip()
        month_num = MONTH_MAP[month]

        for (base, state) in state_names:
            def get_val(col_offset, b=base):
                try:
                    v = str(row[b + col_offset]).strip().replace(",", ".")
                    return float(v) if v not in ("", "nan", ".", "...", "-") else None
                except (IndexError, ValueError):
                    return None

            records.append({
                "date": f"{year}-{month_num}",
                "year": int(float(year)),
                "month": month,
                "month_num": int(month_num),
                "state": state,
                "insolvencies": get_val(0),
                "yoy_change_pct": get_val(2),
                "claims_tsd_eur": get_val(4),
                "employees_affected": get_val(6),
            })

    result = pd.DataFrame(records).dropna(subset=["insolvencies"])
    return result.reset_index(drop=True)

bundesland_clean = clean_bundesland(f"{DATA_FOLDER}/52411-0110_de.csv")
print(f"bundesland_clean: {bundesland_clean.shape[0]} rows, {bundesland_clean['state'].nunique()} states")
bundesland_clean.head()

### 4e. Enrich Industry Data with Business Register

We merge insolvency counts with the number of active businesses per industry from the Unternehmensregister (business register) to calculate insolvency rates — a much more meaningful metric than raw counts.

In [ ]:
# Parse business register
raw_reg = pd.read_csv(f"{DATA_FOLDER}/52111-0003_de.csv", sep=";",
                       encoding="utf-8-sig", header=None,
                       skiprows=9, on_bad_lines="skip")

raw_reg["Jahr"] = raw_reg[0].where(pd.to_numeric(raw_reg[0], errors="coerce").notna()).ffill()
data_rows = raw_reg[raw_reg[0].astype(str).str.startswith("WZ08")]

register_clean = data_rows[["Jahr", 0, 1, 10]].copy()
register_clean.columns = ["Jahr", "Branchencode", "Branche", "Aktive_Unternehmen"]
register_clean["Aktive_Unternehmen"] = pd.to_numeric(register_clean["Aktive_Unternehmen"], errors="coerce")
register_clean["Jahr"] = register_clean["Jahr"].astype(int)

print(f"register_clean: {register_clean.shape[0]} rows, years {sorted(register_clean['Jahr'].unique())}")

### 4f. Corporate Insolvencies with Employee Impact

Extract company-level insolvency counts and affected employee numbers from file `52411-0014`. The file uses the same year → month → procedure-type row structure. We extract the "Insgesamt" (total) row for each month.

In [ ]:
MONTH_MAP_INV = {v: k for k, v in MONTH_MAP.items()}

def clean_corporate(df):
    """Parse corporate insolvency file (52411-0014).
    Structure: year row → month row → procedure rows (eröffnet, mangels Masse, Insgesamt).
    Col 1 = insolvency count, Col 3 = affected employees.
    """
    current_year, current_month = None, None
    records = []

    for _, row in df.iterrows():
        val = str(row[0]).strip()
        
        if val.replace(".", "").isdigit() and len(val) == 4:
            current_year = int(val)
            continue
        if val in MONTH_MAP:
            current_month = val
            continue
        if val == "Insgesamt" and current_year and current_month:
            month_num = MONTH_MAP[current_month]
            records.append({
                "date": f"{current_year}-{month_num}",
                "year": current_year,
                "month": current_month,
                "month_num": int(month_num),
                "insolvencies": pd.to_numeric(row[1], errors="coerce"),
                "employees_affected": pd.to_numeric(row[3], errors="coerce"),
            })

    return pd.DataFrame(records).reset_index(drop=True)

germany_unternehmen = clean_corporate(df_corporate)
print(f"germany_unternehmen: {germany_unternehmen.shape[0]} rows")
print(f"Total corporate insolvencies: {germany_unternehmen['insolvencies'].sum():,.0f}")
print(f"Total employees affected: {germany_unternehmen['employees_affected'].sum():,.0f}")
germany_unternehmen.head()

## 5. Standardize & Rename Columns

Convert all date columns to a consistent `YYYY-MM` string format and rename columns to German for Tableau compatibility.

In [ ]:
# Standardize dates
for df in [germany_clean, industry_clean, size_clean]:
    df["date"] = pd.to_datetime(df["date"]).dt.strftime("%Y-%m")
    df["month_num"] = df["month_num"].astype(int)

# Rename to German column names
germany_clean = germany_clean.rename(columns={
    "date": "Datum", "year": "Jahr", "month": "Monat", "month_num": "Monatsnummer",
    "insolvencies": "Insolvenzen", "yoy_change_pct": "Veraenderung_Vorjahr_Pct",
    "claims_tsd_eur": "Forderungen_Tsd_EUR",
})

bundesland_clean = bundesland_clean.rename(columns={
    "date": "Datum", "year": "Jahr", "month": "Monat", "month_num": "Monatsnummer",
    "state": "Bundesland", "insolvencies": "Insolvenzen",
    "yoy_change_pct": "Veraenderung_Vorjahr_Pct", "claims_tsd_eur": "Forderungen_Tsd_EUR",
    "employees_affected": "Betroffene_Arbeitnehmer",
})

industry_clean = industry_clean.rename(columns={
    "date": "Datum", "year": "Jahr", "month": "Monat", "month_num": "Monatsnummer",
    "industry_code": "Branchencode", "industry": "Branche",
    "opened": "Eroeffnet", "dismissed": "Mangels_Masse_Abgewiesen", "total": "Gesamt",
})

size_clean = size_clean.rename(columns={
    "date": "Datum", "year": "Jahr", "month": "Monat", "month_num": "Monatsnummer",
    "size_class": "Groessenklasse", "insolvencies": "Insolvenzen",
    "claims_tsd_eur": "Forderungen_Tsd_EUR", "employees_affected": "Betroffene_Arbeitnehmer",
})

germany_unternehmen = germany_unternehmen.rename(columns={
    "date": "Datum", "year": "Jahr", "month": "Monat", "month_num": "Monatsnummer",
    "insolvencies": "Unternehmen_Insolvenzen", "employees_affected": "Betroffene_Arbeitnehmer",
})

print("Column rename complete.")
for name, df in [("germany", germany_clean), ("bundesland", bundesland_clean),
                 ("industry", industry_clean), ("size", size_clean),
                 ("unternehmen", germany_unternehmen)]:
    print(f"  {name:15s} -> {df.columns.tolist()}")

## 6. Derived Datasets

### 6a. Industry Insolvency Rates

Add the ratio of dismissed-for-lack-of-assets cases and merge with business register data to compute insolvency rates per active company.

In [ ]:
# Add mangel ratio
industry_clean["mangel_ratio"] = (
    industry_clean["Mangels_Masse_Abgewiesen"] / industry_clean["Eroeffnet"] * 100
).round(1)

# Merge with business register
industry_register = industry_clean.merge(
    register_clean[["Jahr", "Branchencode", "Aktive_Unternehmen"]],
    left_on=["Branchencode", "Jahr"],
    right_on=["Branchencode", "Jahr"],
    how="left"
)
industry_register["Insolvenzrate_Pct"] = (
    industry_register["Eroeffnet"] / industry_register["Aktive_Unternehmen"] * 100
).round(4)

print(f"industry_register: {industry_register.shape[0]} rows")
print(f"Insolvency rate available for {industry_register['Insolvenzrate_Pct'].notna().sum()} rows")

### 6b. Combined National Overview

Merge total insolvency counts with corporate-only counts and affected employees into a single national summary file.

In [ ]:
germany_combined = germany_clean[["Datum", "Jahr", "Monat", "Monatsnummer", "Insolvenzen"]].copy()
germany_combined = germany_combined.rename(columns={"Insolvenzen": "Alle_Insolvenzen"})

germany_combined = germany_combined.merge(
    germany_unternehmen[["Datum", "Unternehmen_Insolvenzen", "Betroffene_Arbeitnehmer"]],
    on="Datum", how="left"
)

print(f"germany_combined: {germany_combined.shape[0]} rows")
germany_combined.head()

## 7. Export Clean Data

Write all cleaned and enriched datasets to CSV for use in Tableau.

In [ ]:
exports = {
    "germany_clean.csv": germany_clean,
    "bundesland_clean.csv": bundesland_clean,
    "industry_clean.csv": industry_clean,
    "industry_register_clean.csv": industry_register,
    "size_clean.csv": size_clean,
    "germany_unternehmen_clean.csv": germany_unternehmen,
    "germany_combined_clean.csv": germany_combined,
}

for filename, df in exports.items():
    path = f"{CLEAN_FOLDER}/{filename}"
    df.to_csv(path, index=False, encoding="utf-8-sig")
    print(f"  {filename:35s} {df.shape[0]:>5} rows x {df.shape[1]:>2} cols")

print(f"\nAll {len(exports)} files exported to {CLEAN_FOLDER}/")

## 8. Data Summary

Quick validation of the exported datasets.

In [ ]:
total_insolvencies = germany_clean["Insolvenzen"].sum()
total_corporate = germany_unternehmen["Unternehmen_Insolvenzen"].sum()
total_employees = germany_unternehmen["Betroffene_Arbeitnehmer"].sum()
worst_year = germany_clean.groupby("Jahr")["Insolvenzen"].sum().idxmax()
worst_year_count = germany_clean.groupby("Jahr")["Insolvenzen"].sum().max()

print("=" * 55)
print("  DATASET SUMMARY (2020-2025)")
print("=" * 55)
print(f"  Total insolvency filings:    {total_insolvencies:>12,.0f}")
print(f"  Corporate insolvencies:      {total_corporate:>12,.0f}")
print(f"  Employees affected:          {total_employees:>12,.0f}")
print(f"  Peak year:                   {worst_year} ({worst_year_count:,.0f} filings)")
print(f"  Federal states covered:      {bundesland_clean['Bundesland'].nunique():>12}")
print(f"  Industry sectors:            {industry_clean['Branchencode'].nunique():>12}")
print(f"  Time span:                   {germany_clean['Datum'].min()} to {germany_clean['Datum'].max()}")
print("=" * 55)
print("\nClean data ready for Tableau visualization.")